# 03. Config/Seed/Submission Ledger와 Inference 재현성 검증

> 작성일: 2026-07-07 KST
> 범위: baseline RandomForest inference에 실행 설정, seed, commit, data manifest hash, submission SHA256, ledger row를 연결한다.
> 안전 원칙: 이 노트북은 실제 제출 CSV, 모델 파일, `outputs/` 산출물을 만들지 않는다. 모든 실행은 synthetic smoke data로만 수행한다.

## 분석 질문

1. 동일한 `config + seed + code commit + data manifest`로 baseline inference를 반복 실행하면 같은 submission hash가 나오는가?
2. ledger row만 보아도 2차 검증자가 어떤 설정과 데이터 버전으로 제출 파일을 만들었는지 복원할 수 있는가?
3. hash가 실제 CSV bytes와 같은 serializer에서 계산되어 byte-level 증거가 되는가?
4. notebook 실행 결과가 다음 작업자인 validator CLI 설계로 자연스럽게 이어지는가?

## Executive Summary

이번 노트북은 제출 운영에서 가장 중요한 **재현 가능한 제출 증거**를 코드로 검증한다. DACON 2차 제출에서는 train/inference 코드, 모델, 제출 파일, 발표 자료가 서로 맞아야 한다. 따라서 제출 CSV 자체뿐 아니라 그 파일을 만든 설정과 데이터 버전, seed, commit, hash를 함께 남겨야 한다.

| 구성요소 | 이번 노트북의 역할 | 운영상 의미 |
|---|---|---|
| `BaselineRunConfig` | 실험 ID, seed, model params, notes를 안정적으로 직렬화 | config hash와 seed 추적 |
| seed | RandomForest와 numpy/Python random 고정 | 같은 seed로 같은 결과를 얻는 근거 |
| commit/data manifest | 코드 버전과 데이터 버전 기록 | ledger row 복구 키 |
| submission SHA256 | 제출 CSV bytes의 fingerprint | canonical CSV bytes와 연결 |
| submission ledger | 제출 파일, 점수, 선택 여부, 메모 기록 | 한 행짜리 DataFrame schema 검증 |

**Decision Box**

| 결정 | 이유 | 상태 |
|---|---|---|
| 실제 공식 데이터 대신 synthetic smoke를 사용 | 노트북 실행이 빠르고 안전하며 실제 제출 CSV를 만들 위험이 없음 | 채택 |
| 기본 실행에서 파일을 쓰지 않음 | 산출물은 CLI/운영 단계에서 명시적으로 생성해야 함 | 채택 |
| hash는 `submission_to_csv_bytes()` 기준으로 계산 | ledger hash와 저장될 파일 bytes가 분리되면 복구 증거가 약해짐 | 채택 |

In [1]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
  sys.path.insert(0, str(SRC_DIR))

from baram.metrics import TARGET_COLS
from baram.reproducibility import (
  SUBMISSION_LEDGER_COLUMNS,
  BaselineRunConfig,
  build_submission_filename,
  ledger_entry_to_frame,
  run_baseline_with_reproducibility,
  submission_csv_sha256,
  submission_to_csv_bytes,
  verify_baseline_inference_reproducibility,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TARGET_COLS:", TARGET_COLS)


PROJECT_ROOT: C:\Users\kik32\workspace\Dacon\2026-BARAM-Wind-Power-Prediction-AI-Competition
TARGET_COLS: ['kpx_group_1', 'kpx_group_2', 'kpx_group_3']


## Synthetic Smoke Fixture 설계

이 synthetic fixture는 실제 대회 데이터의 핵심 구조만 작게 재현한다.

- `train_labels`: `kst_dtm`과 target 3개를 가진다. Group 3에는 결측을 넣어 target별 non-null mask를 검증한다.
- `ldaps/gfs`: `forecast_kst_dtm`, `data_available_kst_dtm`, grid metadata, u/v 기상 변수를 가진다.
- `sample_submission`: `forecast_id`, `forecast_kst_dtm`, target 3개를 가진 제출 template 역할을 한다.

실제 제출 파일은 절대 만들지 않는다. 이 노트북의 목적은 모델 성능이 아니라 API 계약, schema, hash, ledger의 연결을 검증하는 것이다.

In [2]:
def make_weather_frame(times, values):
  rows = []
  for time_index, forecast_time in enumerate(pd.to_datetime(times)):
    available_time = forecast_time - pd.Timedelta(hours=12)
    for grid_id in [1, 2]:
      value = float(values[time_index] + grid_id)
      rows.append(
        {
          "forecast_kst_dtm": forecast_time,
          "data_available_kst_dtm": available_time,
          "grid_id": grid_id,
          "latitude": 37.0 + grid_id * 0.01,
          "longitude": 128.0 + grid_id * 0.01,
          "heightAboveGround_10_10u": value,
          "heightAboveGround_10_10v": value * 2,
        }
      )
  return pd.DataFrame(rows)


def make_label_frame():
  times = pd.date_range("2024-01-01 01:00:00", periods=5, freq="h")
  return pd.DataFrame(
    {
      "kst_dtm": times,
      "kpx_group_1": [1000.0, 3000.0, 5000.0, 7000.0, 9000.0],
      "kpx_group_2": [2000.0, 4000.0, 6000.0, 8000.0, 10000.0],
      "kpx_group_3": [np.nan, np.nan, 2100.0, 5000.0, 8000.0],
    }
  )


def make_sample_submission(times):
  return pd.DataFrame(
    {
      "forecast_id": [f"F{i:03d}" for i in range(len(times))],
      "forecast_kst_dtm": pd.to_datetime(times),
      "kpx_group_1": 0.0,
      "kpx_group_2": 0.0,
      "kpx_group_3": 0.0,
    }
  )


def tiny_run_inputs():
  labels = make_label_frame()
  sample = make_sample_submission(pd.date_range("2025-01-01 01:00:00", periods=2, freq="h"))
  return {
    "train_labels": labels,
    "ldaps_train": make_weather_frame(labels["kst_dtm"], [1, 2, 3, 4, 5]),
    "gfs_train": make_weather_frame(labels["kst_dtm"], [10, 20, 30, 40, 50]),
    "sample_submission": sample,
    "ldaps_test": make_weather_frame(sample["forecast_kst_dtm"], [6, 7]),
    "gfs_test": make_weather_frame(sample["forecast_kst_dtm"], [60, 70]),
  }

inputs = tiny_run_inputs()
display(pd.DataFrame({
  "frame": list(inputs.keys()),
  "rows": [len(frame) for frame in inputs.values()],
  "columns": [len(frame.columns) for frame in inputs.values()],
}))


,frame,rows,columns
0,train_labels,5,4
1,ldaps_train,10,7
2,gfs_train,10,7
3,sample_submission,2,5
4,ldaps_test,4,7
5,gfs_test,4,7


### 해석: smoke fixture가 커버하는 범위

이 fixture는 2행짜리 제출 template만 사용하지만, baseline pipeline의 중요한 경계를 모두 지난다.

1. weather aggregation과 calendar feature 생성
2. target별 non-null label mask
3. RandomForest seed 주입
4. sample submission의 ID/time 보존
5. capacity clip 이후 submission assembly

따라서 여기서 같은 hash가 반복 재현되면 ledger가 baseline inference의 핵심 재현 정보를 담는다는 최소 증거가 된다.

In [3]:
config = BaselineRunConfig(
  experiment_id="rf_smoke",
  seed=7,
  model_params={"n_estimators": np.int64(4), "n_jobs": 1, "when": pd.Timestamp("2026-07-07 09:00:00", tz="Asia/Seoul"), "choices": {"b", "a"}},
  notes="synthetic reproducibility smoke",
)
config_native = BaselineRunConfig(
  experiment_id="rf_smoke",
  seed=7,
  model_params={"choices": ["a", "b"], "n_estimators": 4, "n_jobs": 1, "when": "2026-07-07T09:00:00+09:00"},
  notes="synthetic reproducibility smoke",
)

config_audit = pd.DataFrame([
  {"check": "random_state is forced by seed", "value": config.effective_model_params()["random_state"], "pass": config.effective_model_params()["random_state"] == 7},
  {"check": "stable hash ignores param order/native wrappers", "value": config.config_sha256()[:12], "pass": config.config_sha256() == config_native.config_sha256()},
  {"check": "seed change changes config hash", "value": BaselineRunConfig("rf_smoke", seed=8, model_params={"n_estimators": 4}).config_sha256()[:12], "pass": config.config_sha256() != BaselineRunConfig("rf_smoke", seed=8, model_params={"n_estimators": 4}).config_sha256()},
])
display(config_audit)


,check,value,pass
0,random_state is forced by seed,7,True
1,stable hash ignores param order/native wrappers,989671c4423b,True
2,seed change changes config hash,40b56fd867f1,True


### 해석: config hash의 의미

`config_sha256`은 사람이 읽는 설정 설명이 아니라, 제출을 복구하기 위한 fingerprint다.

- `random_state`는 모델 파라미터에 직접 넣지 않아도 config의 `seed`로 강제된다. seed가 ledger에 남으면 같은 모델 난수 상태를 복원할 수 있다.
- numpy scalar, pandas timestamp, set처럼 JSON이 바로 처리하지 못하는 값도 stable JSON으로 정규화한다. 이 처리를 하지 않으면 같은 설정인데도 객체 표현 차이 때문에 다른 hash가 나올 수 있다.

다음 판단: config hash가 충분하려면 데이터 버전과 코드 버전 hash가 반드시 함께 있어야 한다.

In [4]:
filename = build_submission_filename("exp/a b", "abcdef123456", "2026-07-07T09:00:00+09:00")
submission = inputs["sample_submission"].copy()
canonical_bytes = submission_to_csv_bytes(submission)
sha_from_bytes = hashlib.sha256(canonical_bytes).hexdigest()
sha_from_helper = submission_csv_sha256(submission)

serializer_audit = pd.DataFrame([
  {"check": "filename is path-safe", "value": filename, "pass": filename == "submission_20260707_exp-a-b_abcdef1.csv"},
  {"check": "hash uses canonical bytes", "value": sha_from_helper[:12], "pass": sha_from_bytes == sha_from_helper},
  {"check": "canonical bytes include UTF-8 BOM", "value": canonical_bytes[:3], "pass": canonical_bytes.startswith(b"\xef\xbb\xbf")},
])
display(serializer_audit)


,check,value,pass
0,filename is path-safe,submission_20260707_exp-a-b_abcdef1.csv,True
1,hash uses canonical bytes,8d42fe36c808,True
2,canonical bytes include UTF-8 BOM,b'\xef\xbb\xbf',True


### 해석: filename과 CSV bytes를 함께 고정하는 이유

제출 운영에서 가장 위험한 실수는 DataFrame은 같아 보이는데 실제 저장 bytes가 달라지는 경우다. 줄바꿈, BOM, 날짜 포맷, float 표현 하나만 달라도 SHA256은 달라진다.

이번 설계는 `submission_to_csv_bytes()`를 유일한 serializer로 둔다. 노트북에서 계산한 hash와 실제 저장 파일 hash가 같은 bytes에서 만들어져야, ledger hash가 제출 파일을 증명할 수 있다.

다음 판단: 이 책임은 다음 작업인 submission validator CLI로 이동해야 한다.

In [5]:
with tempfile.TemporaryDirectory() as temp_dir:
  before = sorted(os.listdir(temp_dir))
  old_cwd = Path.cwd()
  os.chdir(temp_dir)
  try:
    result = run_baseline_with_reproducibility(
      config=BaselineRunConfig(
        experiment_id="rf_smoke",
        seed=7,
        model_params={"n_estimators": 4, "n_jobs": 1},
        notes="notebook smoke",
      ),
      **inputs,
      git_commit="abcdef123456",
      data_manifest_sha256="manifest-sha",
      created_at_kst="2026-07-07T09:00:00+09:00",
    )
  finally:
    os.chdir(old_cwd)
  after = sorted(os.listdir(temp_dir))

ledger_entry = result["ledger_entry"]
ledger_frame = ledger_entry_to_frame(ledger_entry)

key_columns = [
  "created_at_kst",
  "experiment_id",
  "git_commit",
  "seed",
  "config_sha256",
  "data_manifest_sha256",
  "submission_filename",
  "submission_sha256",
  "submission_rows",
  "selected",
  "note",
]
display(ledger_frame[key_columns])
print("temporary directory before:", before)
print("temporary directory after :", after)
print("submission shape:", result["baseline_result"]["submission"].shape)


,created_at_kst,experiment_id,git_commit,seed,config_sha256,data_manifest_sha256,submission_filename,submission_sha256,submission_rows,selected,note
0,2026-07-07T09:00:00+09:00,rf_smoke,abcdef123456,7,a38005d7664e4e46b5c48c393dec279e7086c8a2bcc356...,manifest-sha,submission_20260707_rf_smoke_abcdef1.csv,3c3ab54d103ad3d4de79a150853b7a60a54a82bb49b76f...,2,False,notebook smoke


temporary directory before: []
temporary directory after : []
submission shape: (2, 5)


### 해석: ledger row가 담아야 하는 정보

한 ledger row는 나중에 `outputs/submissions/submission_ledger.csv`에 누적될 수 있는 최소 제출 기록이다.

- `created_at_kst`: 제출 또는 후보 생성 시점을 KST로 고정한다.
- `experiment_id`: 실험 목적을 식별한다.
- `git_commit`: 코드 버전을 고정한다.
- `seed`와 `config_sha256`: 모델 설정을 복구한다.
- `data_manifest_sha256`: 원본 데이터 버전을 고정한다.
- `submission_filename`과 `submission_sha256`: 제출 파일의 byte-level fingerprint를 남긴다.

임시 디렉터리 before/after가 비어 있음을 확인해 기본 실행에서 파일을 쓰지 않는다는 점도 함께 검증했다. 이 단계에서는 파일 저장보다 **ledger metadata 생성과 reproducibility check**가 목적이다.

In [6]:
report = verify_baseline_inference_reproducibility(
  config=BaselineRunConfig(
    experiment_id="rf_smoke",
    seed=7,
    model_params={"n_estimators": 4, "n_jobs": 1},
    notes="notebook repeatability check",
  ),
  **inputs,
  git_commit="abcdef123456",
  data_manifest_sha256="manifest-sha",
  repeats=2,
  created_at_kst="2026-07-07T09:00:00+09:00",
)

run_frame = pd.DataFrame(report["runs"])
summary = pd.DataFrame([
  {"check": "two runs requested", "value": report["repeats"], "pass": report["repeats"] == 2},
  {"check": "same submission sha256", "value": run_frame["submission_sha256"].nunique(), "pass": report["is_reproducible"]},
  {"check": "same config sha256", "value": run_frame["config_sha256"].nunique(), "pass": run_frame["config_sha256"].nunique() == 1},
])
display(run_frame)
display(summary)


,run_index,created_at_kst,submission_filename,submission_sha256,config_sha256
0,1,2026-07-07T09:00:00+09:00,submission_20260707_rf_smoke_abcdef1.csv,3c3ab54d103ad3d4de79a150853b7a60a54a82bb49b76f...,fb6975e17b385f5021c9cffe77c33c59702aeceeea3f50...
1,2,2026-07-07T09:00:00+09:00,submission_20260707_rf_smoke_abcdef1.csv,3c3ab54d103ad3d4de79a150853b7a60a54a82bb49b76f...,fb6975e17b385f5021c9cffe77c33c59702aeceeea3f50...


,check,value,pass
0,two runs requested,2,True
1,same submission sha256,1,True
2,same config sha256,1,True


### 해석: 반복 inference 결과

두 번의 실행에서 `submission_sha256`이 같다는 것은 synthetic smoke 범위에서 다음 조건이 유지됐다는 뜻이다.

1. seed가 RandomForest에 주입됐다.
2. feature column order와 imputer 변환 순서가 안정적이다.
3. submission assembly와 CSV serialization이 같은 bytes를 만들었다.
4. ledger row의 핵심 fingerprint가 반복 실행 사이에 유지됐다.

이것이 전체 대회 데이터의 성능을 보장하지는 않는다. 하지만 운영 관점에서는 같은 설정과 같은 데이터에서 같은 제출 artifact를 다시 만들 수 있다는 최소 계약을 만족한다.

In [7]:
commands = [
  [sys.executable, "-m", "pytest", "tests/test_reproducibility.py", "-q"],
  [sys.executable, "-m", "pytest", "-q"],
]

for command in commands:
  completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
  print("$", " ".join(command))
  print(completed.stdout)
  if completed.stderr:
    print(completed.stderr)
  print("returncode:", completed.returncode)
  assert completed.returncode == 0


$ C:\Users\kik32\anaconda3\python.exe -m pytest tests/test_reproducibility.py -q
.........                                                                [100%]
9 passed in 4.85s

returncode: 0


$ C:\Users\kik32\anaconda3\python.exe -m pytest -q
..................................                                       [100%]
34 passed in 15.38s

returncode: 0


## Final Decision Box

| 항목 | 결정 | 근거 | 다음 연결 |
|---|---|---|---|
| Config/seed 계약 | 채택 | seed를 `random_state`에 강제하고 config hash로 추적 | 모든 실험 config에 적용 |
| Submission hash | 채택 | canonical bytes와 SHA256 helper가 같은 결과 | validator CLI에서 같은 bytes 사용 |
| Ledger row | 채택 | 복구에 필요한 필드를 한 행 DataFrame으로 고정 | `outputs/submissions/submission_ledger.csv` 저장 정책 필요 |
| 반복 inference | 채택 | 2회 실행 submission SHA256 동일 | 실제 데이터 smoke와 모델 artifact 검증으로 확장 |
| 기본 무파일쓰기 | 유지 | 임시 디렉터리 before/after가 비어 있음 | CLI에서도 기본 dry-run 유지 |

## 다음 작업 후보

다음 후보는 `submission validator CLI와 실제 train.py/inference.py 진입점 분리`다. 이유는 재현성 ledger가 의미 있으려면 실제 저장 직전에 제출 CSV 검증이 필요하기 때문이다.

1. 이 노트북의 validator 요구사항을 별도 테스트로 내린다.
2. `src/baram/validation.py` 또는 `src/baram/submission.py`에 validator를 TDD로 구현한다.
3. `train.py`와 `inference.py` CLI를 나누고, 제출 CSV 저장은 validator 통과 뒤에만 허용한다.
4. 새 노트북에는 방법론, 실패 사례, 실행 결과, Decision Box를 남긴다.